# 🔵 K-Means Clustering
**Module 1 — Clustering Algorithms**

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.datasets import make_blobs, load_iris
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 2. What is K-Means?
> K-Means partitions **n** observations into **k** clusters where each observation belongs to the cluster with the **nearest mean (centroid)**. It minimizes within-cluster sum of squares (WCSS / Inertia).

**Algorithm:**
1. Initialize k centroids randomly
2. Assign each point to nearest centroid
3. Recompute centroids as cluster mean
4. Repeat steps 2–3 until convergence

**Key Parameters:** `n_clusters`, `init` (k-means++ / random), `max_iter`, `n_init`

## 3. Synthetic Dataset — Make Blobs

In [ ]:
X_blobs, y_true = make_blobs(n_samples=500, centers=4, cluster_std=0.8, random_state=42)
print(f'Dataset shape: {X_blobs.shape}')
print(f'True clusters: {np.unique(y_true)}')

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='tab10', s=30, alpha=0.7)
ax.set_title('Raw Data (4 True Clusters)', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.colorbar(scatter, ax=ax, label='True Label')
plt.tight_layout(); plt.show()

## 4. The Elbow Method — Finding Optimal K

In [ ]:
inertias = []
k_range = range(1, 12)
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_range, inertias, 'bo-', markersize=8, lw=2)
ax.axvline(4, color='red', linestyle='--', lw=1.5, label='Optimal K=4')
ax.fill_between(k_range, inertias, alpha=0.1, color='blue')
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Inertia (WCSS)', fontsize=12)
ax.set_title('Elbow Method — Optimal K Selection', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Silhouette Score — Cluster Validation

In [ ]:
silhouette_scores = []
for k in range(2, 12):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs)
    silhouette_scores.append(silhouette_score(X_blobs, labels))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(2, 12), silhouette_scores, 'gs-', markersize=8, lw=2)
ax.axvline(4, color='red', linestyle='--', lw=1.5, label='Optimal K=4')
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Best Silhouette Score at K=4: {silhouette_scores[2]:.4f}')

## 6. Fit K-Means (K=4)

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, max_iter=300, random_state=42)
labels = kmeans.fit_predict(X_blobs)
centroids = kmeans.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Clustered result
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=30, alpha=0.6)
axes[0].scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, marker='X', zorder=5, label='Centroids')
axes[0].set_title('K-Means Clusters (K=4)', fontsize=13, fontweight='bold')
axes[0].legend()

# True labels
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='tab10', s=30, alpha=0.6)
axes[1].set_title('Ground Truth Labels', fontsize=13, fontweight='bold')

plt.suptitle('K-Means Result vs Ground Truth', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Evaluation Metrics

In [ ]:
sil = silhouette_score(X_blobs, labels)
db  = davies_bouldin_score(X_blobs, labels)
ch  = calinski_harabasz_score(X_blobs, labels)

print('='*45)
print('       K-Means Evaluation Metrics')
print('='*45)
print(f'  Inertia (WCSS)         : {kmeans.inertia_:.2f}')
print(f'  Silhouette Score       : {sil:.4f}  (higher = better, max=1)')
print(f'  Davies-Bouldin Score   : {db:.4f}  (lower  = better, min=0)')
print(f'  Calinski-Harabasz Score: {ch:.2f} (higher = better)')
print(f'  Iterations to converge : {kmeans.n_iter_}')
print('='*45)

## 8. K-Means on Real Dataset — Iris

In [ ]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

km_iris = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_iris = km_iris.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, labels, title in zip(axes, [labels_iris, y_iris], ['K-Means (K=3)', 'Ground Truth']):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set1', s=40, alpha=0.7)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('K-Means on Iris Dataset (PCA Projection)', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Silhouette Score (Iris): {silhouette_score(X_scaled, labels_iris):.4f}')

## 9. Effect of K on Clustering

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, k in enumerate([2, 3, 4, 5, 6, 8]):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lbl = km.fit_predict(X_blobs)
    axes[i].scatter(X_blobs[:, 0], X_blobs[:, 1], c=lbl, cmap='tab10', s=20, alpha=0.6)
    axes[i].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                    c='black', s=150, marker='X', zorder=5)
    sil = silhouette_score(X_blobs, lbl)
    axes[i].set_title(f'K={k}  |  Silhouette={sil:.3f}', fontsize=11, fontweight='bold')
plt.suptitle('K-Means with Different K Values', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Key Takeaways
> - K-Means is **fast and scalable** but assumes **spherical, equal-sized** clusters
> - Sensitive to **outliers** and **initialization**; use `k-means++` always
> - Use **Elbow + Silhouette** together to pick optimal K
> - **Does not work well** with non-convex shapes or varying cluster densities